# 23 · Nonlinear Boundary Test

This notebook follows Notebook 22.

Notebook 22 showed:
- zeta vs Poisson is easy with a linear boundary
- GUE vs Poisson is easy with a linear boundary
- zeta vs GUE is hard with a linear boundary

Now we ask:

> Does a **nonlinear boundary** separate zeta from GUE substantially better?

## Tasks

We compare:
1. **zeta vs GUE**
2. **zeta vs Poisson**
3. **GUE vs Poisson**

using three boundary families:
- linear logistic regression
- quadratic logistic regression
- RBF prototype classifier

## Goal

If nonlinear methods still fail on zeta vs GUE while succeeding on the Poisson tasks, that supports the idea that zeta and GUE share the same low-dimensional local geometry rather than merely evading a simple linear separator.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Common train / test split

We use one fixed train/test partition for each pairwise task.

In [ ]:
zeta_base = zeta_gaps_full[:700]
poisson_base = poisson_gaps_full[:700]
gue_base = gue_gaps_full[:950]

_, zeta_X = build_feature_matrix(zeta_base, k=5)
_, poisson_X = build_feature_matrix(poisson_base, k=5)
_, gue_X = build_feature_matrix(gue_base, k=5)

m = min(len(zeta_X), len(poisson_X), len(gue_X))
zeta_X = zeta_X[:m]
poisson_X = poisson_X[:m]
gue_X = gue_X[:m]

def split_train_test(X, frac=0.6):
    n = len(X)
    cut = int(frac * n)
    return X[:cut], X[cut:]

zeta_train, zeta_test = split_train_test(zeta_X, frac=0.6)
poisson_train, poisson_test = split_train_test(poisson_X, frac=0.6)
gue_train, gue_test = split_train_test(gue_X, frac=0.6)

zeta_train.shape, zeta_test.shape

## Standardization and logistic helpers

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])

    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad

    return w

def predict_proba_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return sigmoid(Xb @ w)

def predict_logistic(X, w, threshold=0.5):
    return (predict_proba_logistic(X, w) >= threshold).astype(int)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

## Feature lifts

Quadratic lift adds pairwise feature interactions.

RBF prototype lift maps each point to similarities with a set of training prototypes.

In [ ]:
def quadratic_features(X):
    x1 = X[:, 0]
    x2 = X[:, 1]
    x3 = X[:, 2]
    return np.column_stack([
        x1, x2, x3,
        x1*x1, x2*x2, x3*x3,
        x1*x2, x1*x3, x2*x3
    ])

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

## Pairwise task runner

In [ ]:
def run_boundary_family_task(name, X0_train, X1_train, X0_test, X1_test):
    m_train = min(len(X0_train), len(X1_train))
    m_test = min(len(X0_test), len(X1_test))

    X_train = np.vstack([X0_train[:m_train], X1_train[:m_train]])
    y_train = np.array([0] * m_train + [1] * m_train)

    X_test = np.vstack([X0_test[:m_test], X1_test[:m_test]])
    y_test = np.array([0] * m_test + [1] * m_test)

    # linear
    Xtr_lin, Xte_lin = standardize_train_test(X_train, X_test)
    w_lin = fit_logistic_regression(Xtr_lin, y_train, lr=0.12, n_steps=3000, reg=1e-4)
    pred_lin = predict_logistic(Xte_lin, w_lin)
    acc_lin = accuracy(y_test, pred_lin)

    # quadratic
    Q_train = quadratic_features(X_train)
    Q_test = quadratic_features(X_test)
    Qtr, Qte = standardize_train_test(Q_train, Q_test)
    w_quad = fit_logistic_regression(Qtr, y_train, lr=0.08, n_steps=3500, reg=1e-4)
    pred_quad = predict_logistic(Qte, w_quad)
    acc_quad = accuracy(y_test, pred_quad)

    # RBF prototype
    Xtr_std, Xte_std = standardize_train_test(X_train, X_test)
    prototypes = choose_prototypes(Xtr_std, n_proto=20)
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    R_test = rbf_features(Xte_std, prototypes, gamma)
    w_rbf = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    pred_rbf = predict_logistic(R_test, w_rbf)
    acc_rbf = accuracy(y_test, pred_rbf)

    return {
        "task": name,
        "linear_accuracy": acc_lin,
        "quadratic_accuracy": acc_quad,
        "rbf_accuracy": acc_rbf,
    }

## Run the three main tasks

In [ ]:
res_zeta_gue = run_boundary_family_task("zeta vs GUE", zeta_train, gue_train, zeta_test, gue_test)
res_zeta_pois = run_boundary_family_task("zeta vs Poisson", zeta_train, poisson_train, zeta_test, poisson_test)
res_gue_pois = run_boundary_family_task("GUE vs Poisson", gue_train, poisson_train, gue_test, poisson_test)

res_zeta_gue, res_zeta_pois, res_gue_pois

## Accuracy comparison

In [ ]:
tasks = [res_zeta_gue, res_zeta_pois, res_gue_pois]
task_names = [t["task"] for t in tasks]
linear_vals = [t["linear_accuracy"] for t in tasks]
quad_vals = [t["quadratic_accuracy"] for t in tasks]
rbf_vals = [t["rbf_accuracy"] for t in tasks]

x = np.arange(len(task_names))
width = 0.24

plt.figure(figsize=(10.5, 5.0))
plt.bar(x - width, linear_vals, width, label="linear")
plt.bar(x, quad_vals, width, label="quadratic")
plt.bar(x + width, rbf_vals, width, label="RBF prototype")
plt.xticks(x, task_names, rotation=20)
plt.ylim(0.45, 1.0)
plt.ylabel("test accuracy")
plt.title("Linear vs nonlinear boundary families")
plt.legend()
plt.tight_layout()
plt.show()

## Improvement over linear baseline

In [ ]:
improvement_summary = {
    "zeta_vs_GUE": {
        "quadratic_minus_linear": res_zeta_gue["quadratic_accuracy"] - res_zeta_gue["linear_accuracy"],
        "rbf_minus_linear": res_zeta_gue["rbf_accuracy"] - res_zeta_gue["linear_accuracy"],
    },
    "zeta_vs_Poisson": {
        "quadratic_minus_linear": res_zeta_pois["quadratic_accuracy"] - res_zeta_pois["linear_accuracy"],
        "rbf_minus_linear": res_zeta_pois["rbf_accuracy"] - res_zeta_pois["linear_accuracy"],
    },
    "GUE_vs_Poisson": {
        "quadratic_minus_linear": res_gue_pois["quadratic_accuracy"] - res_gue_pois["linear_accuracy"],
        "rbf_minus_linear": res_gue_pois["rbf_accuracy"] - res_gue_pois["linear_accuracy"],
    },
}
improvement_summary

## Height-block check for the hardest task

We repeat zeta vs GUE across height blocks and compare linear vs nonlinear performance.

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]

height_results = []

for s, label in zip(block_starts, block_labels):
    zeta_block = zeta_gaps_full[s:s + block_size]
    _, zeta_block_X = build_feature_matrix(zeta_block, k=5)

    m_block = min(len(zeta_block_X), len(gue_X))
    z_block = zeta_block_X[:m_block]
    g_block = gue_X[:m_block]

    z_tr, z_te = split_train_test(z_block, frac=0.6)
    g_tr, g_te = split_train_test(g_block, frac=0.6)

    out = run_boundary_family_task("zeta vs GUE", z_tr, g_tr, z_te, g_te)
    height_results.append({
        "block": label,
        "linear_accuracy": out["linear_accuracy"],
        "quadratic_accuracy": out["quadratic_accuracy"],
        "rbf_accuracy": out["rbf_accuracy"],
    })

height_results

In [ ]:
plt.figure(figsize=(9.8, 4.8))
plt.plot(block_labels, [r["linear_accuracy"] for r in height_results], marker="o", label="linear")
plt.plot(block_labels, [r["quadratic_accuracy"] for r in height_results], marker="o", label="quadratic")
plt.plot(block_labels, [r["rbf_accuracy"] for r in height_results], marker="o", label="RBF prototype")
plt.ylabel("zeta vs GUE accuracy")
plt.ylim(0.45, 0.8)
plt.title("Nonlinear boundary test across height")
plt.legend()
plt.tight_layout()
plt.show()

## Compact summary

In [ ]:
summary = {
    "main_tasks": {
        "zeta_vs_GUE": res_zeta_gue,
        "zeta_vs_Poisson": res_zeta_pois,
        "GUE_vs_Poisson": res_gue_pois,
    },
    "improvement_summary": improvement_summary,
    "height_results": height_results,
}
summary

## Notes

- If nonlinear classifiers improve the Poisson tasks much more than the zeta-vs-GUE task, that supports the interpretation that the zeta/GUE overlap is structural, not just a missed linear separator.
- If the zeta-vs-GUE task remains close to chance even after quadratic and RBF lifts, then the local feature geometry remains genuinely difficult to separate.
- This notebook uses lightweight nonlinear lifts rather than a full hyperparameter sweep.

## Next directions

A natural next notebook could do one of these:

1. bootstrap the nonlinear-accuracy differences  
2. compare several feature sets under the same nonlinear protocol  
3. train on GUE vs Poisson and evaluate confidence directly on zeta vs GUE  
4. visualize nonlinear confidence surfaces in PCA space